# Chapter 15: Language Models and the Transformer
##  English-to-Spanish Translation

This notebook walks through two approaches to **sequence-to-sequence (seq2seq) learning** for machine translation:

1. **GRU-based encoder–decoder** — a recurrent neural network approach
2. **Transformer-based encoder–decoder** — the attention-based architecture that powers modern LLMs

Both models are trained on the same English→Spanish dataset so you can directly compare how each architecture works.

---
**Learning objectives:**
- Understand how to prepare a parallel translation dataset
- Build and train a GRU-based seq2seq model
- Implement a Transformer encoder and decoder from scratch
- Understand positional embeddings and why they matter
- Generate translations token-by-token at inference time

---
## Step 1: Install and Configure Dependencies

We use **Keras** (a high-level deep learning framework) with the **JAX** backend for efficient computation.
We also set the backend *before* importing Keras — this must be done via an environment variable.

In [ ]:
# Install or upgrade Keras to the latest version
!pip install keras --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.2 MB/s eta 0:00:00


In [ ]:
import os

# Set the Keras backend to JAX before importing Keras.
# Valid options are: "jax", "tensorflow", "torch"
# This must be done before any Keras import.
os.environ["KERAS_BACKEND"] = "jax"

---
## Step 2: Load the English–Spanish Translation Dataset

We download the **spa-eng** dataset — a collection of sentence pairs used for machine translation.
Each line in the file contains one English sentence and its Spanish translation, separated by a tab.

We add special `[start]` and `[end]` tokens to every Spanish sentence. These tokens are critical:
- **`[start]`** signals to the decoder: "begin generating the translation now"
- **`[end]`** signals: "the translation is complete — stop generating"

In [ ]:
import keras
import pathlib

# Download and extract the Spanish-English sentence pairs dataset
zip_path = keras.utils.get_file(
    origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# Parse the file into (english, spanish) pairs.
# Each line is: english_sentence \t spanish_sentence
# We wrap Spanish sentences with [start] and [end] tokens.

with open(text_path) as f:
    lines = f.read().split("\n")[:-1]  # remove trailing empty line

text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"  # wrap with boundary tokens
    text_pairs.append((english, spanish))

# Preview a sample pair
import random
random.choice(text_pairs)

('I have lost my wallet.', '[start] He perdido mi monedero. [end]')

---
## Step 3: Split the Data into Train / Validation / Test Sets

We randomly shuffle all sentence pairs and split them into three sets:
- **Train (70%)** — used to update model weights
- **Validation (15%)** — used to monitor performance during training (not used for updates)
- **Test (15%)** — held out entirely; used to evaluate final model performance

Shuffling prevents the model from learning any ordering artifact in the original file.

In [ ]:
import random

# Shuffle the full dataset randomly
random.shuffle(text_pairs)

# Compute split sizes
val_samples = int(0.15 * len(text_pairs))      # 15% for validation
train_samples = len(text_pairs) - 2 * val_samples  # remaining 70% for training

train_pairs = text_pairs[:train_samples]
val_pairs   = text_pairs[train_samples : train_samples + val_samples]
test_pairs  = text_pairs[train_samples + val_samples:]

print(f"Training pairs:   {len(train_pairs)}")
print(f"Validation pairs: {len(val_pairs)}")
print(f"Test pairs:       {len(test_pairs)}")

Training pairs:   83276
Validation pairs: 17844
Test pairs:       17844


---
## Step 4: Tokenization — Converting Text to Token IDs

Neural networks cannot process raw text — they need numbers. **Tokenization** maps each word (or character) to an integer ID from a fixed vocabulary.

We build **two separate tokenizers** — one for English, one for Spanish — because they have different vocabularies.

Key parameters:
- `max_tokens=15000` — vocabulary size (top 15k most frequent words)
- `output_sequence_length` — all sequences are padded or truncated to this length
- `standardize` — a custom function to lowercase and remove punctuation from Spanish text (the `¿` character needs special handling)

The Spanish tokenizer is given `sequence_length + 1` tokens because during training we need to:
- **Input** positions 0…N-1 (the decoder sees these)
- **Predict** positions 1…N (the target labels are shifted by 1)

In [ ]:
import string
import re
import tensorflow as tf
from keras import layers

# Build the character set to strip from Spanish text.
# We keep [ and ] because they're part of our [start]/[end] tokens.
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    """Lowercase and remove punctuation from Spanish text."""
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, f"[{re.escape(strip_chars)}]", "")

# Vocabulary and sequence settings
vocab_size = 15000   # number of unique tokens per language
sequence_length = 20 # max number of tokens per sentence

# English tokenizer — uses default standardization (lowercase, strip punctuation)
english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

# Spanish tokenizer — uses custom standardization to preserve [start]/[end]
# +1 length so we can split input and target labels with a 1-step shift
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

# Build vocabularies from training data only
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

print("English vocabulary size:", english_tokenizer.vocabulary_size())
print("Spanish vocabulary size:", spanish_tokenizer.vocabulary_size())

English vocabulary size: 12058
Spanish vocabulary size: 15000


---
## Step 5: Build the Training Dataset Pipeline

We create a `tf.data.Dataset` pipeline that:
1. Tokenizes both English and Spanish sentences
2. Creates the **teacher forcing** setup for the decoder:
   - **Input to decoder**: Spanish tokens from position 0 to N-1 (`spa[:, :-1]`)
   - **Labels (targets)**: Spanish tokens from position 1 to N (`spa[:, 1:]`)
   - This means: given the previous token, predict the next one
3. Creates **sample weights** to ignore padding tokens (token ID = 0) during loss calculation

**Teacher forcing** means we feed the *true* previous token to the decoder during training, not its own (possibly wrong) prediction. This speeds up training significantly.

In [ ]:
batch_size = 64

def format_dataset(eng, spa):
    """Tokenize a batch and set up teacher forcing inputs/labels."""
    eng = english_tokenizer(eng)              # shape: (batch, 20)
    spa = spanish_tokenizer(spa)              # shape: (batch, 21)

    # Decoder input: all tokens except the last one
    # Decoder label: all tokens except the first one (shifted by 1)
    features = {"english": eng, "spanish": spa[:, :-1]}  # (batch, 20)
    labels = spa[:, 1:]                                    # (batch, 20)

    # Sample weights: ignore padding tokens (id=0) in the loss
    sample_weights = labels != 0
    return features, labels, sample_weights


def make_dataset(pairs):
    """Create a batched, shuffled, cached dataset from sentence pairs."""
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()  # cache for faster subsequent epochs


train_ds = make_dataset(train_pairs)
val_ds   = make_dataset(val_pairs)

# Inspect the shape of one batch
inputs, targets, sample_weights = next(iter(train_ds))
print("English input shape:", inputs["english"].shape)   # (64, 20)
print("Spanish input shape:", inputs["spanish"].shape)   # (64, 20)
print("Labels shape:",        targets.shape)              # (64, 20)

English input shape: (64, 20)
Spanish input shape: (64, 20)
Labels shape: (64, 20)


---
## Part A: GRU-Based Sequence-to-Sequence Model

The first approach uses **Gated Recurrent Units (GRUs)** — a type of recurrent neural network designed to handle sequences.

### Architecture Overview

```
English sentence  →  [Encoder GRU]  →  context vector (final hidden state)
                                               ↓
Spanish [start] + partial sentence  →  [Decoder GRU initialized with context]  →  predicted next token
```

- **Encoder**: reads the full English sentence and compresses it into a single fixed-size vector (the hidden state)
- **Decoder**: uses that vector as its starting state, then generates Spanish tokens one at a time

We use a **Bidirectional GRU** for the encoder so it reads the sentence in both left→right and right→left directions, capturing context from both sides. The two directions are summed together (`merge_mode="sum"`).

### Step 6A: Build the GRU Encoder

In [ ]:
embed_dim  = 256   # dimensionality of token embeddings
hidden_dim = 1024  # size of the GRU hidden state

# --- ENCODER ---
# Input: a sequence of English token IDs
source = keras.Input(shape=(None,), dtype="int32", name="english")

# Embedding: map each integer token ID → a dense vector of size embed_dim
# mask_zero=True tells the model to ignore padding tokens (ID=0)
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)

# Bidirectional GRU: reads sequence forwards AND backwards
# merge_mode="sum" adds the two directions together into one vector
# No return_sequences — we only need the final hidden state (the context)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)  # shape: (batch, hidden_dim)

print("Encoder output shape:", encoder_output.shape)

Encoder output shape: (None, 1024)


### Step 6B: Build the GRU Decoder and Assemble the Full Model

In [ ]:
# --- DECODER ---
# Input: partial Spanish sentence (during training, this is the teacher-forced input)
target = keras.Input(shape=(None,), dtype="int32", name="spanish")

# Embed Spanish tokens just like English
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)

# Unidirectional GRU decoder:
# - return_sequences=True: we want a prediction at every time step, not just the last
# - initial_state=encoder_output: initialize the decoder with the encoder's context vector
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)

# Dropout for regularization — randomly zero 50% of activations during training
x = layers.Dropout(0.5)(x)

# Output layer: at each step, predict a probability over all Spanish vocabulary words
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

# Assemble the full encoder-decoder model
# Inputs: English sentence + partial Spanish sentence
# Output: probability distribution over next Spanish token at each position
seq2seq_rnn = keras.Model([source, target], target_predictions)
seq2seq_rnn.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding (Embedding) │ (None, None, 256) │   3,840,000 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, None)      │           0 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_1           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding[0][0],   │
│ (Bidirectional)       │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_1 (GRU)           │ (None, None,      │   3,938,304 │ embedding_1[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout (Dropout)     │ (None, None,      │           0 │ gru_1[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense (Dense)         │ (None, None,      │  15,375,000 │ dropout[0][0]      │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

### Step 7: Train the GRU Model

We use:
- **Adam optimizer** — adaptive learning rate, well-suited for seq2seq
- **Sparse categorical crossentropy** — standard loss for multi-class token prediction ("sparse" means labels are integer IDs, not one-hot vectors)
- **Sample weights** — ensure padding positions don't contribute to the loss

In [ ]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],  # accuracy weighted by sample_weights
)

seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 136s 100ms/step - accuracy: 0.3614 - loss: 3.6493 - val_accuracy: 0.5054 - val_loss: 2.4668
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 129s 99ms/step - accuracy: 0.5394 - loss: 2.2379 - val_accuracy: 0.5968 - val_loss: 1.8784
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 101ms/step - accuracy: 0.6244 - loss: 1.6154 - val_accuracy: 0.6303 - val_loss: 1.6777
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 101ms/step - accuracy: 0.6831 - loss: 1.2327 - val_accuracy: 0.6434 - val_loss: 1.6240
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 100ms/step - accuracy: 0.7285 - loss: 0.9821 - val_accuracy: 0.6527 - val_loss: 1.6111
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 100ms/step - accuracy: 0.7637 - loss: 0.8194 - val_accuracy: 0.6552 - val_loss: 1.6370
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 100ms/step - accuracy: 0.7880 - loss: 0.7142 - val_accuracy: 0.6545 - val_loss: 1.6914
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 100ms/step - acc

### Step 8: Generate Translations with the GRU Model

At **inference time**, we no longer have the true Spanish sentence to feed as input. Instead, we generate one token at a time using **greedy decoding**:

1. Start with `"[start]"` as the decoder input
2. Run the model → get a probability distribution over the next token
3. Pick the token with the highest probability (`argmax`)
4. Append it to the decoded sentence and repeat
5. Stop when `"[end]"` is produced or we reach `sequence_length`

Note: this is greedy decoding. In practice, **beam search** (keeping the top-k candidates at each step) produces better results.

In [ ]:
import numpy as np

# Build a lookup table: token ID → Spanish word
spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation_rnn(input_sentence):
    """Translate an English sentence to Spanish using greedy decoding."""
    # Tokenize the English input
    tokenized_input_sentence = english_tokenizer([input_sentence])

    # Start the decoded sentence with the [start] token
    decoded_sentence = "[start]"

    for i in range(sequence_length):
        # Tokenize the partial Spanish sentence decoded so far
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]

        # Get prediction probabilities for the next token
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)

        # Greedy: pick the token with highest probability at position i
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]

        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":  # stop when end token is generated
            break

    return decoded_sentence


# Test on 5 random sentences from the held-out test set
test_eng_texts = [pair[0] for pair in test_pairs]
print("GRU Model Translations:")
print("=" * 50)
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print("English:", input_sentence)
    print("Spanish:", generate_translation_rnn(input_sentence))

GRU Model Translations:
-
English: I don't like either of the boys.
Spanish: [start] no me gustan los chicos de los demás [end]
-
English: All his endeavors proved unsuccessful.
Spanish: [start] todos se [UNK] su [UNK] [end]
-
English: Tom felt like a fish out of water.
Spanish: [start] tom sintió como un pez de agua [end]
-
English: The fireman soon put the fire out.
Spanish: [start] el fuego se decidió al fuego [end]
-
English: Can you prove that what you said is true?
Spanish: [start] puede usted describir lo que sea que no sea cierto [end]


---
## Part B: Transformer-Based Sequence-to-Sequence Model

The **Transformer** architecture (Vaswani et al., 2017 — "Attention Is All You Need") replaced RNNs with **self-attention mechanisms**. Key advantages:

- **Parallelizable**: unlike GRUs, all positions are processed simultaneously
- **Direct connections**: any two tokens can attend to each other regardless of distance
- **Scalable**: more parameters → better performance (basis of GPT, BERT, etc.)

### Architecture Overview

```
English tokens → [Positional Embedding] → [TransformerEncoder] → encoder_output
                                                                        ↓
Spanish tokens → [Positional Embedding] → [TransformerDecoder (with cross-attention)] → predictions
```

We will implement both the **TransformerEncoder** and **TransformerDecoder** as custom Keras layers.

### Step 9: Implement the Transformer Encoder Block

The Transformer Encoder processes the **source** (English) sequence. Each encoder block contains:

1. **Multi-Head Self-Attention**: each token attends to all other tokens in the same sequence
   - Multiple attention "heads" learn different types of relationships in parallel
2. **Residual connection + Layer Normalization**: adds the input back to the attention output to prevent vanishing gradients
3. **Feed-Forward Network**: two dense layers applied independently at each position
4. **Another residual + LayerNorm**

The **attention mask** (`source_mask`) ensures padding positions are ignored — we don't want the model attending to zero-padded tokens.

In [ ]:
class TransformerEncoder(keras.Layer):
    """
    A single Transformer encoder block.

    Applies multi-head self-attention followed by a position-wise
    feed-forward network, with residual connections and layer normalization
    after each sub-layer.

    Args:
        hidden_dim:       Dimensionality of the token representations (model dimension).
        intermediate_dim: Hidden size of the feed-forward network (usually 4x hidden_dim).
        num_heads:        Number of attention heads.
    """

    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads  # per-head key/query/value dimensionality

        # --- Sub-layer 1: Multi-Head Self-Attention ---
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()

        # --- Sub-layer 2: Position-wise Feed-Forward Network ---
        # Two Dense layers: expand to intermediate_dim then project back to hidden_dim
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        """
        Args:
            source:      Token representations, shape (batch, seq_len, hidden_dim).
            source_mask: Boolean mask, shape (batch, seq_len). True = real token, False = padding.
        Returns:
            Encoded representations, shape (batch, seq_len, hidden_dim).
        """
        # Save input for residual connection
        residual = x = source

        # Reshape mask for attention: (batch, 1, seq_len)
        # MultiHeadAttention expects (batch, query_len, key_len) → broadcast with [None, :]
        mask = source_mask[:, None, :]

        # Self-attention: each position attends to all positions
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)

        # Residual connection + layer norm
        x = x + residual
        x = self.self_attention_layernorm(x)

        # Feed-forward sub-layer with another residual connection
        residual = x
        x = self.feed_forward_1(x)  # expand
        x = self.feed_forward_2(x)  # project back
        x = x + residual
        x = self.feed_forward_layernorm(x)

        return x

### Step 10: Implement the Transformer Decoder Block

The Transformer Decoder processes the **target** (Spanish) sequence and has **three sub-layers**:

1. **Masked Multi-Head Self-Attention** (`use_causal_mask=True`):
   - Each Spanish token can only attend to *previous* tokens (not future ones)
   - This is the **causal mask** — essential because at inference time the model generates left-to-right

2. **Cross-Attention** (encoder–decoder attention):
   - Queries come from the decoder (Spanish tokens)
   - Keys and Values come from the encoder output (English representation)
   - This is how the decoder "looks at" the source sentence

3. **Feed-Forward Network** — same structure as in the encoder

In [ ]:
class TransformerDecoder(keras.Layer):
    """
    A single Transformer decoder block.

    Contains:
      1. Masked self-attention (causal — looks only at past positions)
      2. Cross-attention (attends to the encoder's output)
      3. Feed-forward network
    Each sub-layer has a residual connection and layer normalization.

    Args:
        hidden_dim:       Dimensionality of token representations.
        intermediate_dim: Hidden size of the feed-forward network.
        num_heads:        Number of attention heads.
    """

    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads

        # --- Sub-layer 1: Masked Self-Attention (causal) ---
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()

        # --- Sub-layer 2: Cross-Attention (decoder attends to encoder output) ---
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()

        # --- Sub-layer 3: Feed-Forward Network ---
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        """
        Args:
            target:      Target token representations (Spanish), shape (batch, tgt_len, hidden_dim).
            source:      Encoder output (English), shape (batch, src_len, hidden_dim).
            source_mask: Mask for encoder output, shape (batch, src_len).
        Returns:
            Decoded representations, shape (batch, tgt_len, hidden_dim).
        """
        # --- Masked Self-Attention ---
        # use_causal_mask=True prevents each position from attending to future positions
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)

        # --- Cross-Attention ---
        # Query = Spanish representations, Key & Value = English (encoder) output
        residual = x
        mask = source_mask[:, None, :]  # shape: (batch, 1, src_len)
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)

        # --- Feed-Forward Network ---
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)

        return x

### Step 11: Train the Basic Transformer (without Positional Embedding)

First, we train a Transformer **without positional information** as a baseline. We'll see why this is limiting and fix it in the next step.

The model architecture:
- English tokens → `Embedding` → `TransformerEncoder` → `encoder_output`
- Spanish tokens → `Embedding` → `TransformerDecoder(encoder_output)` → `Dense(softmax)` → token predictions

Hyperparameters:
- `hidden_dim=256`: each token is represented as a 256-dimensional vector
- `intermediate_dim=2048`: the feed-forward network expands to 2048 before projecting back
- `num_heads=8`: 8 attention heads, each with `256/8 = 32` dimensions

In [ ]:
hidden_dim       = 256
intermediate_dim = 2048
num_heads        = 8

# --- Encoder path ---
source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,  # True for real tokens, False for padding
)

# --- Decoder path ---
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

# Assemble the full Transformer model
transformer = keras.Model([source, target], target_predictions)
transformer.summary(line_length=80)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_2           │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_2           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_encoder   │ (None, None, 256) │   1,315,072 │ embedding_2[0][0], │
│ (TransformerEncoder)  │                   │             │ not_equal_2[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_3           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_3           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_decoder   │ (None, None, 256) │   1,578,752 │ transformer_encod… │
│ (TransformerDecoder)  │                   │             │ not_equal_3[0][0], │
│                       │                   │             │ embedding_3[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_4 (Dropout)   │ (None, None, 256) │           0 │ transformer_decod… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_5 (Dense)       │ (None, None,      │   3,855,000 │ dropout_4[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 14,428,824 (55.04 MB)

 Trainable params: 14,428,824 (55.04 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 48s 31ms/step - accuracy: 0.3595 - loss: 1.4917 - val_accuracy: 0.4948 - val_loss: 1.0381
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - accuracy: 0.5298 - loss: 0.9778 - val_accuracy: 0.5700 - val_loss: 0.8305
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - accuracy: 0.6010 - loss: 0.7689 - val_accuracy: 0.5997 - val_loss: 0.7583
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 32s 25ms/step - accuracy: 0.6448 - loss: 0.6434 - val_accuracy: 0.6148 - val_loss: 0.7273
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - accuracy: 0.6774 - loss: 0.5560 - val_accuracy: 0.6231 - val_loss: 0.7163
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - accuracy: 0.7038 - loss: 0.4903 - val_accuracy: 0.6274 - val_loss: 0.7148
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 32s 25ms/step - accuracy: 0.7265 - loss: 0.4393 - val_accuracy: 0.6276 - val_loss: 0.7290
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 32s 24ms/step - accuracy: 0.7454 -

---
## Step 12: Adding Positional Embeddings

### Why does the Transformer need positional information?

Unlike GRUs, which process tokens **sequentially** and inherently track order, the Transformer processes all tokens **in parallel**. This means it has no built-in sense of which token comes first, second, or third.

Without positional information:
> "The cat sat on the mat" and "Mat the on sat cat the" would look the same to the model!

### Solution: Positional Embedding

We add a **learned position embedding** to each token embedding. This creates a unique representation for each position (0, 1, 2, ...) that gets added on top of the token meaning.

```
Final embedding = Token Embedding + Position Embedding
```

- `token_embeddings`: what the token *means*
- `position_embeddings`: *where* the token appears in the sequence

Both are learned during training.

In [ ]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    """
    Combines token embeddings with learned positional embeddings.

    This gives the model information about both:
      - WHAT each token means (token embedding)
      - WHERE each token appears in the sequence (position embedding)

    Args:
        sequence_length: Maximum number of positions to embed.
        input_dim:       Vocabulary size (number of unique tokens).
        output_dim:      Embedding dimensionality.
    """

    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        # Learns a vector for each token in the vocabulary
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        # Learns a vector for each position 0, 1, 2, ..., sequence_length-1
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        # Generate position indices: [0, 1, 2, ..., seq_len-1] for each example
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1

        embedded_tokens    = self.token_embeddings(inputs)     # token meaning
        embedded_positions = self.position_embeddings(positions)  # position info

        # Sum: each position gets a unique combination of content + location
        return embedded_tokens + embedded_positions

### Step 13: Build and Train the Full Transformer with Positional Embeddings

This is the complete, proper Transformer model. We simply replace the plain `Embedding` layers with `PositionalEmbedding` layers. Everything else stays the same.

We train for more epochs (30) because the positional information significantly increases what the model can learn.

In [ ]:
hidden_dim       = 256
intermediate_dim = 2056
num_heads        = 8

# --- Encoder path with positional embeddings ---
source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

# --- Decoder path with positional embeddings ---
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

# Final model
transformer = keras.Model([source, target], target_predictions)

transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

### Step 14: Generate Translations with the Transformer

Inference is identical to the GRU model — greedy token-by-token decoding. The key difference is that the Transformer processes the full target sequence at once each step, but we still read off predictions position by position.

Note the `tokenized_target_sentence[:, :-1]` — we trim the last token because the Spanish tokenizer added one extra position (for the shifted label setup during training).

In [ ]:
import numpy as np

# Build token-ID-to-word lookup for Spanish
spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation_transformer(input_sentence):
    """Translate an English sentence using the Transformer with greedy decoding."""
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"

    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])

        # Trim the extra position added by the +1 sequence length in the tokenizer
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]

        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)

        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence


# Test on 5 random sentences from the test set
test_eng_texts = [pair[0] for pair in test_pairs]
print("Transformer Model Translations:")
print("=" * 50)
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print("English:", input_sentence)
    print("Spanish:", generate_translation_transformer(input_sentence))

Transformer Model Translations:
-
English: There must be some misunderstanding.
Spanish: [start] debe haber algo de malentendido [end]
-
English: I didn't know how to swim very well.
Spanish: [start] no sabía nadar demasiado bien [end]
-
English: Report to the emergency room.
Spanish: [start] [UNK] a la habitación de él [end]
-
English: I thought Tom was a Canadian.
Spanish: [start] pensé que tom era canadiense [end]
-
English: He described the accident in detail to the police.
Spanish: [start] Él describió el accidente en detalle a la policía [end]


---
## Summary: GRU vs. Transformer

| Feature | GRU Seq2Seq | Transformer |
|---|---|---||
| **Processes sequence** | Sequentially (left→right) | In parallel (all positions at once) |
| **Long-range dependencies** | Harder (information bottleneck) | Easy (direct attention between any two tokens) |
| **Training speed** | Slower (sequential) | Faster (parallelizable) |
| **Positional information** | Built-in (recurrence = order) | Must be added explicitly |
| **Context vector** | Single fixed-size vector | Full sequence of encoder outputs |
| **Scalability** | Limited | Foundation of GPT, BERT, T5, etc. |

The Transformer is more complex to implement but significantly more powerful and scalable — which is why virtually all modern large language models are Transformer-based.